# Optimizador de Pricing de Préstamos

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Predecir elasticidad de precio** con `SNOWFLAKE.ML.CLASSIFICATION`
2. **Simular escenarios de pricing** con diferentes tipos de interés
3. **Optimizar margen vs volumen** con análisis de sensibilidad
4. **Generar recomendaciones** de pricing con `CORTEX.COMPLETE()`
5. **Construir dashboard** de simulación de pricing

---

## Lo Que Construirás

Un **optimizador de pricing** que encuentra el tipo de interés óptimo:
- Modelo de aceptación por segmento y tipo de interés
- Simulación de escenarios: agresivo, competitivo, conservador
- Análisis de margen vs volumen
- Recomendaciones IA por segmento

**Valor de Negocio**: Maximizar el margen manteniendo volumen competitivo.

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.CLASSIFICATION` — Modelo de aceptación
- `CORTEX.COMPLETE()` — Recomendaciones de pricing
- Simulaciones SQL — Escenarios de tipo de interés
- `POWER()` — Cálculos de TIR y cuota

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS PRICING_PRESTAMOS_DB;
CREATE SCHEMA IF NOT EXISTS PRICING_PRESTAMOS_DB.PUBLIC;
USE SCHEMA PRICING_PRESTAMOS_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Datos de Solicitudes con Pricing

### Qué Vamos a Crear

- **10.000 solicitudes** con tipo de interés ofertado y resultado (aceptado/rechazado)
- **Relación inversamente proporcional**: A mayor tipo de interés, menor aceptación
- **Segmentos**: Premium (menos sensible al precio), Particular (más sensible)

In [ ]:
CREATE OR REPLACE TABLE SOLICITUDES_PRICING (
    solicitud_id VARCHAR(10) PRIMARY KEY,
    segmento VARCHAR(20),
    importe DECIMAL(12,2),
    plazo_meses INTEGER,
    tipo_interes_ofertado DECIMAL(5,3),
    ingresos_mensuales DECIMAL(10,2),
    dti_ratio FLOAT,
    score_credito INTEGER,
    competidores_consultados INTEGER,
    aceptado BOOLEAN
);

INSERT INTO SOLICITUDES_PRICING
SELECT
    'SOL' || LPAD(SEQ4()::STRING, 6, '0'),
    ARRAY_CONSTRUCT('Premium','Particular','Joven','Empresa')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    ROUND(UNIFORM(5000, 200000, RANDOM()), 2),
    ARRAY_CONSTRUCT(12, 24, 36, 48, 60, 84)[UNIFORM(0,5,RANDOM())]::INT,
    ROUND(UNIFORM(2.0, 12.0, RANDOM()), 3),
    ROUND(UNIFORM(1500, 10000, RANDOM()), 2),
    ROUND(UNIFORM(0.1, 0.6, RANDOM()), 2),
    UNIFORM(400, 850, RANDOM()),
    UNIFORM(0, 5, RANDOM()),
    CASE
        WHEN UNIFORM(2.0, 12.0, RANDOM()) < 5.0 AND UNIFORM(0,100,RANDOM()) < 70 THEN TRUE
        WHEN UNIFORM(2.0, 12.0, RANDOM()) < 8.0 AND UNIFORM(0,100,RANDOM()) < 40 THEN TRUE
        WHEN UNIFORM(0,100,RANDOM()) < 15 THEN TRUE
        ELSE FALSE
    END
FROM TABLE(GENERATOR(ROWCOUNT => 10000));

SELECT aceptado, COUNT(*), ROUND(AVG(tipo_interes_ofertado),2) AS tipo_medio
FROM SOLICITUDES_PRICING GROUP BY 1;

---

## Paso 3: Modelo de Aceptación

### Predecir Probabilidad de Aceptación

Entrenamos un modelo que predice si el cliente aceptará la oferta dado el tipo de interés.

In [ ]:
CREATE OR REPLACE TABLE FEATURES_PRICING AS
SELECT
    solicitud_id,
    CASE segmento WHEN 'Premium' THEN 3 WHEN 'Empresa' THEN 2 WHEN 'Particular' THEN 1 ELSE 0 END::FLOAT AS nivel_segmento,
    importe::FLOAT,
    plazo_meses::FLOAT,
    tipo_interes_ofertado::FLOAT,
    ingresos_mensuales::FLOAT,
    dti_ratio::FLOAT,
    score_credito::FLOAT,
    competidores_consultados::FLOAT,
    (importe * tipo_interes_ofertado / 1200 * POWER(1 + tipo_interes_ofertado / 1200, plazo_meses)) / 
        NULLIF(POWER(1 + tipo_interes_ofertado / 1200, plazo_meses) - 1, 0)::FLOAT AS cuota_mensual,
    aceptado
FROM SOLICITUDES_PRICING;

CREATE OR REPLACE TABLE TRAIN_PRICING AS SELECT * FROM FEATURES_PRICING SAMPLE (80);
CREATE OR REPLACE TABLE TEST_PRICING AS SELECT * FROM FEATURES_PRICING WHERE solicitud_id NOT IN (SELECT solicitud_id FROM TRAIN_PRICING);

CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION modelo_aceptacion(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_PRICING'),
    TARGET_COLNAME => 'ACEPTADO'
);

In [ ]:
CALL modelo_aceptacion!SHOW_EVALUATION_METRICS();
CALL modelo_aceptacion!SHOW_FEATURE_IMPORTANCE();

---

## Paso 4: Simulación de Escenarios de Pricing

### Tres Estrategias

| Estrategia | Tipo Interés | Margen | Volumen Esperado |
|---|---|---|---|
| Agresiva | -1% vs actual | Bajo | Alto |
| Competitiva | Actual | Medio | Medio |
| Conservadora | +1% vs actual | Alto | Bajo |

In [ ]:
-- Simular escenarios
CREATE OR REPLACE TABLE SIMULACION_PRICING AS
WITH base AS (SELECT AVG(tipo_interes_ofertado) AS tipo_medio FROM SOLICITUDES_PRICING)
SELECT
    'Agresivo' AS escenario,
    ROUND(b.tipo_medio - 1, 2) AS tipo_simulado,
    COUNT(*) AS solicitudes_simuladas,
    SUM(CASE WHEN s.tipo_interes_ofertado <= b.tipo_medio - 1 AND s.aceptado THEN 1 ELSE 0 END) AS aceptaciones_estimadas,
    ROUND(SUM(CASE WHEN s.tipo_interes_ofertado <= b.tipo_medio - 1 THEN s.importe ELSE 0 END) * (b.tipo_medio - 1) / 100, 0) AS margen_estimado
FROM SOLICITUDES_PRICING s, base b GROUP BY b.tipo_medio
UNION ALL
SELECT 'Competitivo', ROUND(b.tipo_medio, 2),
    COUNT(*),
    SUM(CASE WHEN s.aceptado THEN 1 ELSE 0 END),
    ROUND(SUM(CASE WHEN s.aceptado THEN s.importe ELSE 0 END) * b.tipo_medio / 100, 0)
FROM SOLICITUDES_PRICING s, base b GROUP BY b.tipo_medio
UNION ALL
SELECT 'Conservador', ROUND(b.tipo_medio + 1, 2),
    COUNT(*),
    SUM(CASE WHEN s.tipo_interes_ofertado >= b.tipo_medio + 1 AND s.aceptado THEN 1 ELSE 0 END),
    ROUND(SUM(CASE WHEN s.tipo_interes_ofertado >= b.tipo_medio + 1 THEN s.importe ELSE 0 END) * (b.tipo_medio + 1) / 100, 0)
FROM SOLICITUDES_PRICING s, base b GROUP BY b.tipo_medio;

SELECT * FROM SIMULACION_PRICING;

---

## Paso 5: Dashboard de Pricing

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Optimizador de Pricing de Préstamos')
st.markdown('### Simulación de Escenarios')

df = session.sql('SELECT * FROM SIMULACION_PRICING').to_pandas()
for _, row in df.iterrows():
    st.subheader(f"Escenario: {row['ESCENARIO']}")
    c1, c2, c3 = st.columns(3)
    c1.metric('Tipo Interés', f"{row['TIPO_SIMULADO']}%")
    c2.metric('Aceptaciones', f"{row['ACEPTACIONES_ESTIMADAS']:,}")
    c3.metric('Margen Estimado', f"{row['MARGEN_ESTIMADO']:,.0f}€")

st.markdown('---')
st.subheader('Comparativa')
st.bar_chart(df.set_index('ESCENARIO')[['ACEPTACIONES_ESTIMADAS','MARGEN_ESTIMADO']])

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS PRICING_PRESTAMOS_DB;